# Lecture 3 · Notebook 3 — A real neural LM: scoring and sampling with GPT-2, filling blanks with BERT
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_3_sampling_gpt2_and_bert.ipynb)

**Companion to [Chapters 1, 7 and 9 of the video lecture](https://nlp-course-cuhksz.github.io/lecture03.html#p1-s11)**. GPT-2 (124M parameters, 2019) is a small decoder-only
language model of the kind that now dominates the field; DistilBERT is a compact version of BERT's masked LM. In this notebook you will:

1. score text with GPT-2: the monkey's string vs. a real sentence (slide 7), and real vs. shuffled word order,
2. look at the **next-word distribution**, like the phone keyboard on slide 8,
3. run the **autoregressive loop** by hand, with and without the **KV cache** (slide 11),
4. sample with **temperature, top-k and top-p** — the same spin-the-wheel, with 50,257 slices,
5. test the **fill-in-the-blank** examples from slides 17–20 and BERT's `[MASK]` (slides 109 and 119), and
6. see that BERT gives *bank* a different vector in different contexts (slides 100–101).

Runs on CPU in about 3–5 minutes; downloads about 0.8 GB (GPT-2 + DistilBERT).

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, math, time
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
print("GPT-2 on", device, f"- {sum(p.numel() for p in gpt2.parameters()) / 1e6:.0f}M parameters, vocabulary {len(tok):,}")

## 1 · A language model scores text (slides 7 and 13)
By the chain rule, $\log P(w_1..w_n) = \sum_i \log P(w_i \mid w_{<i})$. GPT-2 gives us every conditional in one forward pass.
We report the total log-probability and the per-token perplexity.

In [ ]:
@torch.no_grad()
def score(text):
    ids = tok(tok.bos_token + text, return_tensors="pt").input_ids.to(device)   # <|endoftext|> as the start token
    logp = F.log_softmax(gpt2(ids).logits[0, :-1], -1)
    tok_lp = logp.gather(1, ids[0, 1:, None])[:, 0]
    return tok_lp.sum().item(), math.exp(-tok_lp.mean().item()), len(tok_lp)

for s in ["sfklkljf fskjhf", "ChatGPT is all you need", "need you all is ChatGPT",
          "The students opened their books.", "The students opened their exams."]:
    lp, ppl, n = score(s)
    print(f"{s!r:36s} log P = {lp:7.1f}   ({n} tokens, perplexity {ppl:8.1f})")

Compare the **total** log-probabilities: "ChatGPT is all you need" is far more likely than the monkey's string. The per-token
perplexity can mislead here, because the monkey's string is split into more (and individually less surprising) sub-word tokens —
perplexity is only comparable on the same text and the same tokenizer (slide 72).

Same words, different order → very different probability: unlike a unigram model (slide 37), GPT-2 cares about word order.
Now the Alice passage from the slides, original vs. shuffled:

In [ ]:
import random
alice = ("Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: "
         "once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, "
         "'and what is the use of a book,' thought Alice 'without pictures or conversation?'")
words = alice.split(); random.Random(0).shuffle(words)
print(f"original perplexity: {score(alice)[1]:.1f}")
print(f"shuffled perplexity: {score(' '.join(words))[1]:.1f}")

## 2 · The next-word distribution (slide 8)

In [ ]:
@torch.no_grad()
def next_words(prefix, k=8):
    ids = tok(prefix, return_tensors="pt").input_ids.to(device)
    p = F.softmax(gpt2(ids).logits[0, -1], -1)
    top = p.topk(k)
    return ", ".join(f"{tok.decode(int(i))!r} {v:.2f}" for v, i in zip(top.values, top.indices))

for prefix in ["I've", "I think, therefore I", "The students opened their", "Language models are"]:
    print(f"{prefix!r:30s} -> {next_words(prefix)}")

## 3 · Autoregressive generation by hand, and the KV cache (slide 11)
Generation is a loop: predict one token, append it, run again. Without a cache, every step re-reads the whole prefix. With the
**KV cache**, the keys and values of earlier positions are stored, so each step only processes the one new token.

In [ ]:
@torch.no_grad()
def greedy(prompt, steps=40, use_cache=True):
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    out, past, x = ids, None, ids
    for _ in range(steps):
        if use_cache:
            r = gpt2(x, past_key_values=past, use_cache=True)
            past = r.past_key_values
        else:
            r = gpt2(out)                           # re-encode the full prefix every step
        nxt = r.logits[0, -1].argmax().view(1, 1)
        out = torch.cat([out, nxt], 1)
        x = nxt                                     # with a cache, feed only the new token
    return tok.decode(out[0])

texts = {}
for cache in [False, True]:
    t0 = time.time(); texts[cache] = greedy("Language models are", use_cache=cache)
    print(f"use_cache={cache}: {time.time() - t0:.2f}s")
print("same text:", texts[False] == texts[True])
print(texts[True])

Both runs produce the same text (greedy decoding is deterministic); the cached one is faster, and the gap grows with length.
Greedy decoding also tends to **repeat itself** — one reason we sample instead.

## 4 · Sampling: temperature, top-k, top-p
The same roulette wheel as in the n-gram notebook, just with 50,257 slices. Temperature reshapes the wheel; top-k keeps only the *k*
largest slices; top-p keeps the smallest set of slices whose total probability is at least *p*.

In [ ]:
prompt = "In a shocking finding, scientists discovered a herd of unicorns"
enc = tok(prompt, return_tensors="pt").to(device)
ids = enc.input_ids
settings = [dict(do_sample=False),
            dict(do_sample=True, temperature=0.7, top_k=0),
            dict(do_sample=True, temperature=1.0, top_k=0),
            dict(do_sample=True, temperature=1.0, top_k=40),
            dict(do_sample=True, temperature=1.0, top_p=0.9, top_k=0)]
for i, s in enumerate(settings):
    torch.manual_seed(i)
    out = gpt2.generate(**enc, max_new_tokens=30, pad_token_id=tok.eos_token_id, **s)
    print(s, "\n   ", tok.decode(out[0, ids.shape[1]:]).replace("\n", " "), "\n")

In [ ]:
# How flat is the wheel? Entropy of the next-token distribution at different temperatures
with torch.no_grad():
    logits = gpt2(ids).logits[0, -1]
for T in [0.5, 1.0, 1.5]:
    p = F.softmax(logits / T, -1)
    ent = -(p * p.clamp_min(1e-12).log()).sum().item()
    print(f"T={T}: entropy {ent:.2f} nats  (= a uniform wheel with {math.exp(ent):.0f} slices); top-1 prob {p.max():.2f}")

## 5 · What does predicting the next word teach? (slides 17–20)
GPT-2 is small, so it will not get all of these — try them and see which ones it gets.

In [ ]:
tests = [("I was thinking about the sequence that goes 1, 1, 2, 3, 5, 8, 13, 21,", "math"),
         ("Overall, the value I got from the two hours watching it was the sum total of the popcorn and the drink. The movie was", "sentiment"),
         ("The woman walked across the street, checking for traffic over", "coreference"),
         ("I went to the ocean to see the fish, turtles, seals, and", "world knowledge")]
for prefix, skill in tests:
    print(f"[{skill}] ...{prefix[-45:]!r}\n    -> {next_words(prefix, k=5)}")

## 6 · BERT's masked LM (slides 109 and 119)
BERT is not a left-to-right model: it fills a `[MASK]` using context on **both** sides. We use DistilBERT, a smaller distilled BERT
with the same masked-LM head.

In [ ]:
from transformers import pipeline, AutoModel
fill = pipeline("fill-mask", model="distilbert-base-uncased", device=0 if device == "cuda" else -1)
for s in ["Paris is the [MASK] of France.",
          "The man went to the [MASK] to buy a gallon of milk.",
          "The woman walked across the street, checking for traffic over [MASK] shoulder.",
          "Overall, the value I got from the two hours watching it was the sum total of the popcorn and the drink. The movie was [MASK]."]:
    print(s, "\n   ", ", ".join(f"{r['token_str']} {r['score']:.2f}" for r in fill(s, top_k=5)))

## 7 · Contextualized word vectors: which *bank*? (slides 100–101)
Word2vec gives *bank* one vector. BERT's last layer gives a different vector each time, depending on the sentence.

In [ ]:
btok = fill.tokenizer
bert = AutoModel.from_pretrained("distilbert-base-uncased").to(device).eval()

@torch.no_grad()
def vec(sentence, word="bank"):
    x = btok(sentence, return_tensors="pt").to(device)
    pos = x.input_ids[0].tolist().index(btok.convert_tokens_to_ids(word))
    return bert(**x).last_hidden_state[0, pos].cpu()

sents = ["We sat on the river bank and watched the water.",
         "The fisherman walked along the muddy bank of the stream.",
         "I need to deposit this cheque at the bank today.",
         "The bank raised its interest rates again."]
V = torch.stack([vec(s) for s in sents])
sim = F.normalize(V, dim=1) @ F.normalize(V, dim=1).T
print("cosine similarity of 'bank' across sentences:")
for i, s in enumerate(sents):
    print(f"  s{i + 1} " + " ".join(f"{sim[i, j]:.2f}" for j in range(len(sents))) + f"   {s}")

The two *river* banks are closer to each other than to the two *money* banks: the vector depends on the context. A static embedding
would give 1.00 everywhere.

## Exercises
1. In section 1, find a short grammatical sentence that GPT-2 scores *lower* than an ungrammatical one. What does this tell you about
   "probability" vs. "grammaticality" (slide 13)?
2. In section 4, generate 5 samples with `top_k=1`. Why are they all the same? Which setting would you use for a story, and which for
   code?
3. Replace `"gpt2"` with `"gpt2-medium"` (355M) and rerun section 5. Which fill-in-the-blank skills improve with scale (slide 123)?
4. Write a masked sentence where the right answer depends on a word **to the right** of the mask. Can GPT-2's next-word distribution solve
   it? Can BERT? (Slides 107–108.)

**Back to the [video lecture](https://nlp-course-cuhksz.github.io/lecture03.html)** · [Notebook 1](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_1_ngram_language_models.ipynb) · [Notebook 2](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture03/lecture03_2_fixed_window_neural_lm.ipynb)